# Greenhouse time-series EDA

온실별 **수집 기간이 다르므로**, 한 figure 안에서 온실을 **같은 날짜 축에 겹쳐 그리지 않고**  
변수마다 **온실 1개 = 서브플롯 1행**으로 세로 배치합니다.

- x축: 각 온실의 실제 datetime  
- y축: `Tout`, `Tin`, `RHin`, `CO2`, `Radout`, `Radcum`  
- 아래 셀: 변수별 시각화 (matplotlib, 영어 라벨)

In [ ]:
# 공통 설정: 경로, 변수 매핑, xlsx 로더
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

# 데이터 폴더 (이 노트북과 동일 디렉터리)
DATA_DIR = Path(".").resolve()

# 영문 변수명 → 원본 한글 컬럼명
VAR_COL_MAP = {
    "Tout": "외부-외부온도",
    "Tin": "내부-내부온도",
    "RHin": "내부-내부습도",
    "CO2": "내부-내부CO2",
    "Radout": "외부-외부일사량",
    "Radcum": "외부-누적일사량(가공_외부)",
}

# y축 단위 (matplotlib 라벨용, 영어만 사용)
VAR_UNITS = {
    "Tout": "degC",
    "Tin": "degC",
    "RHin": "%",
    "CO2": "ppm",
    "Radout": "W/m2",
    "Radcum": "MJ/m2",
}

VARIABLES = list(VAR_COL_MAP.keys())


def load_greenhouse_xlsx(filepath: Path) -> pd.DataFrame:
    """xlsx를 읽고 수집일을 DatetimeIndex로 설정한 DataFrame을 반환한다."""
    df = pd.read_excel(filepath, dtype=str)

    date_col = None
    for c in df.columns:
        if "수집일" in c or "date" in c.lower() or "time" in c.lower():
            date_col = c
            break
    if date_col is None:
        date_col = df.columns[0]

    def _parse_dt(s):
        s = str(s).strip()
        if len(s) >= 15 and s[10] == " " and ":" not in s[11:]:
            s = s[:11] + s[11:13] + ":" + s[13:15] + ":" + s[15:17]
        return pd.to_datetime(s, errors="coerce")

    df[date_col] = df[date_col].apply(_parse_dt)
    df = df.dropna(subset=[date_col]).set_index(date_col).sort_index()
    df.index.name = "datetime"

    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    return df


def list_pf_files(data_dir: Path) -> list[Path]:
    """PF_*.xlsx 파일만 정렬하여 반환한다."""
    return sorted(data_dir.glob("PF_*.xlsx"))


# 온실 ID → 전처리된 시계열
greenhouse_series: dict[str, pd.DataFrame] = {}
for fp in list_pf_files(DATA_DIR):
    gh_id = fp.stem
    greenhouse_series[gh_id] = load_greenhouse_xlsx(fp)
    t0, t1 = greenhouse_series[gh_id].index.min(), greenhouse_series[gh_id].index.max()
    print(f"{gh_id}: {t0.date()} ~ {t1.date()}  (n={len(greenhouse_series[gh_id]):,})")

print(f"\nLoaded {len(greenhouse_series)} greenhouses.")

In [ ]:
# 변수별 시계열 플롯: 온실마다 서브플롯 1행 (세로 스택)
def plot_variable_timeseries(var_name: str, row_height: float = 2.2) -> None:
    """한 변수에 대해 온실별 서브플롯을 세로로 쌓아 시계열을 그린다."""
    if var_name not in VAR_COL_MAP:
        raise ValueError(f"Unknown variable: {var_name}")

    col_kr = VAR_COL_MAP[var_name]
    gh_ids = sorted(greenhouse_series.keys())
    n_gh = len(gh_ids)

    fig, axes = plt.subplots(
        n_gh,
        1,
        figsize=(14, row_height * n_gh),
        sharex=False,
        squeeze=False,
    )
    axes = axes.ravel()

    for ax, gh_id in zip(axes, gh_ids):
        df = greenhouse_series[gh_id]
        if col_kr not in df.columns:
            ax.text(0.5, 0.5, f"Missing column: {col_kr}", ha="center", va="center", transform=ax.transAxes)
            ax.set_ylabel(f"{var_name} ({VAR_UNITS[var_name]})")
            ax.text(0.01, 0.95, gh_id, transform=ax.transAxes, va="top", fontsize=10, fontweight="bold")
            continue

        y = df[col_kr]
        ax.plot(df.index, y, linewidth=0.6, color="#2166AC")
        ax.set_ylabel(f"{var_name} ({VAR_UNITS[var_name]})")
        ax.text(0.01, 0.95, gh_id, transform=ax.transAxes, va="top", fontsize=10, fontweight="bold")
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
        ax.tick_params(axis="x", rotation=30)
        ax.grid(True, alpha=0.25)

    axes[-1].set_xlabel("Date")
    fig.tight_layout()
    plt.show()

In [ ]:
# External temperature (Tout)
plot_variable_timeseries("Tout")

In [ ]:
# Internal temperature (Tin)
plot_variable_timeseries("Tin")

In [ ]:
# Internal relative humidity (RHin)
plot_variable_timeseries("RHin")

In [ ]:
# Internal CO2
plot_variable_timeseries("CO2")

In [ ]:
# External solar radiation (Radout)
plot_variable_timeseries("Radout")

In [ ]:
# Cumulative external solar radiation (Radcum)
plot_variable_timeseries("Radcum")